# Spectral EDA

`data/train.csv` を使って、今回の回帰課題に関係が深い4つの可視化を確認します。

- 含水率の KDE
- 代表サンプルのスペクトル折れ線
- 樹種ごとの平均スペクトル
- 樹種ごとの各波数と含水率の相関係数

可視化はすべて Plotly を使い、波数軸は近赤外スペクトルで見慣れた向きになるよう高波数側から低波数側へ表示します。

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from sklearn.neighbors import KernelDensity

px.defaults.template = "plotly_white"


def find_project_root(start: Path) -> Path:
    """Return the repository root by searching for pyproject.toml."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml が見つかりません。")



def spectral_columns(frame: pd.DataFrame) -> list[str]:
    """Return spectral column names excluding metadata columns."""
    metadata_cols = {"sample number", "species number", "樹種", "含水率"}
    return [col for col in frame.columns if col not in metadata_cols]



def compute_kde_curve(values: pd.Series, points: int = 512) -> pd.DataFrame:
    """Estimate a one-dimensional KDE curve for the target variable."""
    numeric = values.dropna().to_numpy(dtype=float)
    if numeric.size < 2:
        raise ValueError("KDE を描くには 2 件以上のデータが必要です。")

    std = np.std(numeric, ddof=1)
    iqr = np.subtract(*np.percentile(numeric, [75, 25]))
    sigma = min(std, iqr / 1.34) if iqr > 0 else std
    bandwidth = 0.9 * sigma * numeric.size ** (-1 / 5) if sigma > 0 else 1.0
    bandwidth = max(bandwidth, 1e-3)

    grid_min, grid_max = numeric.min(), numeric.max()
    padding = max((grid_max - grid_min) * 0.05, bandwidth * 2)
    grid = np.linspace(grid_min - padding, grid_max + padding, points)

    model = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
    model.fit(numeric.reshape(-1, 1))
    density = np.exp(model.score_samples(grid.reshape(-1, 1)))

    return pd.DataFrame({"含水率": grid, "density": density})



def apply_spectral_layout(fig: go.Figure, title: str, yaxis_title: str) -> go.Figure:
    """Apply a shared layout for spectral line plots."""
    fig.update_layout(
        title=title,
        height=650,
        hovermode="x unified",
        legend_title_text="",
        margin=dict(t=80, r=20, b=40, l=60),
    )
    fig.update_xaxes(title="波数 (cm^-1)", autorange="reversed", showgrid=True)
    fig.update_yaxes(title=yaxis_title, showgrid=True)
    return fig


PROJECT_ROOT = find_project_root(Path.cwd())
TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "test.csv"

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, encoding="cp932")
test_df = pd.read_csv(TEST_PATH, encoding="cp932")
spectral_cols = spectral_columns(train_df)
wavenumbers = np.asarray(spectral_cols, dtype=float)

if spectral_cols != spectral_columns(test_df):
    raise ValueError("train と test のスペクトル列が一致しません。")

species_counts = (
    train_df.groupby(["species number", "樹種"], as_index=False)
    .size()
    .rename(columns={"size": "サンプル数"})
    .sort_values("species number")
)
species_order = species_counts["樹種"].tolist()

summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "レコード数": [len(train_df), len(test_df)],
        "樹種数": [train_df["樹種"].nunique(), test_df["樹種"].nunique()],
        "波数数": [len(spectral_cols), len(spectral_cols)],
        "含水率の最小値": [train_df["含水率"].min(), np.nan],
        "含水率の中央値": [train_df["含水率"].median(), np.nan],
        "含水率の最大値": [train_df["含水率"].max(), np.nan],
    }
)

print("データ概要")
print(summary.round(3).to_string(index=False))
print()
print("train の樹種ごとのサンプル数")
print(species_counts.to_string(index=False))

## 1. 含水率の KDE

まずは目的変数の分布を滑らかに確認します。平均値と中央値も重ねて、分布の偏りを見やすくしています。

In [ ]:
kde_curve = compute_kde_curve(train_df["含水率"])
mean_moisture = train_df["含水率"].mean()
median_moisture = train_df["含水率"].median()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=kde_curve["含水率"],
        y=kde_curve["density"],
        mode="lines",
        line=dict(color="#636EFA", width=3),
        fill="tozeroy",
        fillcolor="rgba(99, 110, 250, 0.18)",
        hovertemplate="含水率=%{x:.2f}<br>密度=%{y:.4f}<extra></extra>",
        name="KDE",
    )
)
fig.add_trace(
    go.Scatter(
        x=train_df["含水率"],
        y=np.zeros(len(train_df)),
        mode="markers",
        marker=dict(size=6, opacity=0.35, color="#636EFA"),
        hovertemplate="含水率=%{x:.2f}<extra></extra>",
        showlegend=False,
    )
)
fig.add_vline(x=mean_moisture, line_dash="dash", line_color="#EF553B")
fig.add_vline(x=median_moisture, line_dash="dot", line_color="#00CC96")
fig.add_annotation(
    x=mean_moisture,
    y=float(kde_curve["density"].max()) * 0.92,
    text=f"平均: {mean_moisture:.2f}",
    showarrow=False,
    font=dict(color="#EF553B"),
    bgcolor="rgba(255,255,255,0.8)",
)
fig.add_annotation(
    x=median_moisture,
    y=float(kde_curve["density"].max()) * 0.82,
    text=f"中央値: {median_moisture:.2f}",
    showarrow=False,
    font=dict(color="#00A878"),
    bgcolor="rgba(255,255,255,0.8)",
)
fig.update_layout(
    title="含水率の KDE",
    height=550,
    hovermode="x unified",
    showlegend=False,
    margin=dict(t=80, r=20, b=40, l=60),
)
fig.update_xaxes(title="含水率")
fig.update_yaxes(title="密度")
fig.show()

## 2. 代表サンプルのスペクトル

全件を重ねる代わりに、含水率の低い側から高い側まで均等に代表サンプルを選び、スペクトル形状の違いを確認します。凡例は `sample number` です。

In [ ]:
n_representative_samples = 12
sample_indices = np.linspace(0, len(train_df) - 1, n_representative_samples, dtype=int)
representative_samples = train_df.sort_values("含水率").iloc[sample_indices].copy()
palette = sample_colorscale("Viridis", np.linspace(0.05, 0.95, len(representative_samples)))

fig = go.Figure()
for color, (_, row) in zip(palette, representative_samples.iterrows()):
    customdata = np.column_stack(
        [
            np.repeat(int(row["sample number"]), len(wavenumbers)),
            np.repeat(row["樹種"], len(wavenumbers)),
            np.repeat(float(row["含水率"]), len(wavenumbers)),
        ]
    )
    fig.add_trace(
        go.Scatter(
            x=wavenumbers,
            y=row[spectral_cols].to_numpy(dtype=float),
            mode="lines",
            line=dict(color=color, width=2),
            name=str(int(row["sample number"])),
            customdata=customdata,
            hovertemplate=(
                "sample number=%{customdata[0]}<br>"
                "樹種=%{customdata[1]}<br>"
                "含水率=%{customdata[2]:.2f}<br>"
                "波数=%{x:.1f}<br>"
                "吸光度=%{y:.4f}<extra></extra>"
            ),
        )
    )

apply_spectral_layout(fig, "代表サンプルのスペクトル", "吸光度")
fig.update_layout(
    height=700,
    legend=dict(yanchor="top", y=1.0, xanchor="left", x=1.02),
    margin=dict(t=80, r=160, b=40, l=60),
)
fig.show()

representative_samples[["sample number", "樹種", "含水率"]].reset_index(drop=True)

## 3. 樹種ごとの平均スペクトル

各樹種のスペクトルを平均化して、樹種間でベースラインや吸収形状にどれくらい違いがあるかを俯瞰します。

In [ ]:
mean_spectra = (
    train_df.groupby(["species number", "樹種"], as_index=False)[spectral_cols]
    .mean()
    .sort_values("species number")
)

palette = px.colors.qualitative.Dark24
fig = go.Figure()
for i, (_, row) in enumerate(mean_spectra.iterrows()):
    fig.add_trace(
        go.Scatter(
            x=wavenumbers,
            y=row[spectral_cols].to_numpy(dtype=float),
            mode="lines",
            line=dict(color=palette[i % len(palette)], width=2),
            name=row["樹種"],
            hovertemplate=(
                f"樹種={row['樹種']}<br>"
                "波数=%{x:.1f}<br>"
                "平均吸光度=%{y:.4f}<extra></extra>"
            ),
        )
    )

apply_spectral_layout(fig, "樹種ごとの平均スペクトル", "平均吸光度")
fig.update_layout(
    height=750,
    legend=dict(yanchor="top", y=1.0, xanchor="left", x=1.02),
    margin=dict(t=80, r=180, b=40, l=60),
)
fig.show()

## 4. 樹種ごとの各波数と含水率の相関係数

19樹種を一度に重ねると読みにくいため、小分けのファセット表示にしています。縦軸は共通で `-1` から `1` にそろえ、樹種間で比較しやすくしています。

In [ ]:
correlation_frames = []
species_order = []

for _, species_row in species_counts.iterrows():
    species_name = species_row["樹種"]
    group = train_df.loc[train_df["樹種"] == species_name]
    correlations = group[spectral_cols].corrwith(group["含水率"])
    delta_correlations = correlations - correlations.median()
    species_order.append(species_name)
    correlation_frames.append(
        pd.DataFrame(
            {
                "樹種": species_name,
                "波数": wavenumbers,
                "Δr": delta_correlations.to_numpy(dtype=float),
                "相関係数": correlations.to_numpy(dtype=float),
                "サンプル数": len(group),
            }
        )
    )

correlation_df = pd.concat(correlation_frames, ignore_index=True)
correlation_df["樹種"] = pd.Categorical(
    correlation_df["樹種"],
    categories=species_order,
    ordered=True,
)
correlation_df = correlation_df.sort_values(["樹種", "波数"])

delta_abs_max = float(np.nanmax(np.abs(correlation_df["Δr"].to_numpy(dtype=float))))

y_limit = max(0.02, min(0.3, delta_abs_max * 1.05))

fig = px.line(
    correlation_df,
    x="波数",
    y="Δr",
    color="樹種",
    category_orders={"樹種": species_order},
    height=750,
)
fig.add_hline(y=0.0, line_dash="dash", line_color="gray", opacity=0.8)
fig.update_traces(
    hovertemplate=(
        "樹種=%{fullData.name}<br>"
        "波数=%{x:.1f}<br>"
        "Δr=%{y:.4f}<extra></extra>"
    )
)
fig.update_xaxes(title="波数 (cm^-1)", autorange="reversed", showgrid=True)
fig.update_yaxes(title="Δr (r - median(r))", range=[-y_limit, y_limit], showgrid=True)
fig.update_layout(
    title="樹種ごとの各波数と含水率の Δr",
    hovermode="x unified",
    legend_title_text="",
    margin=dict(t=80, r=20, b=40, l=60),
)
fig.show()

## 5. 樹種ごとの含水率分布

`train` データについて、樹種ごとの含水率レンジとばらつきを箱ひげ図で確認します。樹種ごとの分布が大きく異なる場合、モデルが含水率そのものだけでなく樹種差も強く利用する可能性があります。

In [ ]:
fig = px.box(
    train_df,
    x="樹種",
    y="含水率",
    color="樹種",
    category_orders={"樹種": species_order},
    points=False,
    height=650,
)
fig.update_traces(boxmean=True)
fig.update_layout(
    title="樹種ごとの含水率分布 (train)",
    showlegend=False,
    margin=dict(t=80, r=20, b=40, l=60),
)
fig.update_xaxes(title="樹種", tickangle=-45)
fig.update_yaxes(title="含水率", showgrid=True)
fig.show()

## 6. train と test の差

ここでは `train` と `test` の違いを2つの観点で見ます。

- 樹種構成の差
- 平均スペクトル形状の差

樹種比率や平均スペクトルが大きくずれている場合、`train` で学習したモデルを `test` にそのまま適用すると性能が落ちる可能性があります。

In [ ]:
train_species_counts = (
    train_df.groupby(["species number", "樹種"], as_index=False)
    .size()
    .rename(columns={"size": "サンプル数"})
    .assign(dataset="train")
)
test_species_counts = (
    test_df.groupby(["species number", "樹種"], as_index=False)
    .size()
    .rename(columns={"size": "サンプル数"})
    .assign(dataset="test")
)
split_species_counts = pd.concat(
    [train_species_counts, test_species_counts],
    ignore_index=True,
).sort_values(["species number", "dataset"])

fig = px.bar(
    split_species_counts,
    x="樹種",
    y="サンプル数",
    color="dataset",
    barmode="group",
    category_orders={"樹種": species_order, "dataset": ["train", "test"]},
    color_discrete_map={"train": "#636EFA", "test": "#EF553B"},
    height=550,
)
fig.update_layout(
    title="樹種構成の train / test 比較",
    legend_title_text="",
    margin=dict(t=80, r=20, b=40, l=60),
)
fig.update_xaxes(title="樹種", tickangle=-45)
fig.update_yaxes(title="サンプル数", showgrid=True)
fig.show()

train_mean_spectrum = train_df[spectral_cols].mean().to_numpy(dtype=float)
test_mean_spectrum = test_df[spectral_cols].mean().to_numpy(dtype=float)
mean_spectrum_diff = test_mean_spectrum - train_mean_spectrum

diff_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.68, 0.32],
    subplot_titles=(
        "train / test の平均スペクトル",
        "平均スペクトル差 (test - train)",
    ),
)
diff_fig.add_trace(
    go.Scatter(
        x=wavenumbers,
        y=train_mean_spectrum,
        mode="lines",
        line=dict(color="#636EFA", width=2.5),
        name="train",
        hovertemplate="dataset=train<br>波数=%{x:.1f}<br>平均吸光度=%{y:.4f}<extra></extra>",
    ),
    row=1,
    col=1,
)
diff_fig.add_trace(
    go.Scatter(
        x=wavenumbers,
        y=test_mean_spectrum,
        mode="lines",
        line=dict(color="#EF553B", width=2.5),
        name="test",
        hovertemplate="dataset=test<br>波数=%{x:.1f}<br>平均吸光度=%{y:.4f}<extra></extra>",
    ),
    row=1,
    col=1,
)
diff_fig.add_trace(
    go.Scatter(
        x=wavenumbers,
        y=mean_spectrum_diff,
        mode="lines",
        line=dict(color="#00CC96", width=2),
        name="test - train",
        hovertemplate="波数=%{x:.1f}<br>差分=%{y:.5f}<extra></extra>",
    ),
    row=2,
    col=1,
)
diff_fig.add_hline(y=0.0, line_dash="dash", line_color="gray", row=2, col=1)
diff_fig.update_layout(
    title="train と test の平均スペクトル差",
    height=800,
    hovermode="x unified",
    legend_title_text="",
    margin=dict(t=100, r=20, b=40, l=60),
)
diff_fig.update_xaxes(title="波数 (cm^-1)", autorange="reversed", showgrid=True, row=1, col=1)
diff_fig.update_xaxes(title="波数 (cm^-1)", autorange="reversed", showgrid=True, row=2, col=1)
diff_fig.update_yaxes(title="平均吸光度", showgrid=True, row=1, col=1)
diff_fig.update_yaxes(title="差分", showgrid=True, row=2, col=1)
diff_fig.show()